# RSA, Digital Signatures, and Digital Envelopes

These examples use the running scenario of St. Isidore Hospital. They are teaching examples: understand the mechanism, then prefer well-reviewed libraries and current protocols in production.

## Goal

Public-key cryptography is slower than symmetric encryption but solves different problems: sending a secret to someone whose public key you know, and verifying who signed a message.

In [ ]:
from cryptography.hazmat.primitives.asymmetric import rsa, padding
from cryptography.hazmat.primitives import hashes, serialization
from cryptography.hazmat.primitives.ciphers.aead import AESGCM
import os

doctor_private = rsa.generate_private_key(public_exponent=65537, key_size=2048)  # Doctor's signing key.
doctor_public = doctor_private.public_key()  # Verifiers can know this key.
hospital_private = rsa.generate_private_key(public_exponent=65537, key_size=2048)  # Recipient secret.
hospital_public = hospital_private.public_key()  # Senders use this to protect data for the hospital.

In [ ]:
message = b"Prescription approval for patient 2048"
signature = doctor_private.sign(
    message,
    # PSS is a randomized RSA signature padding scheme.
    padding.PSS(mgf=padding.MGF1(hashes.SHA256()), salt_length=padding.PSS.MAX_LENGTH),
    hashes.SHA256(),
)

doctor_public.verify(
    signature,
    message,
    # Verification repeats the padding/hash checks with the public key.
    padding.PSS(mgf=padding.MGF1(hashes.SHA256()), salt_length=padding.PSS.MAX_LENGTH),
    hashes.SHA256(),
)
print("Signature verified")

## Digital Envelope

The message is encrypted with a fresh AES key. The AES key is then encrypted with the recipient public key.

In [ ]:
clinical_file = b"Discharge summary: diagnosis, therapy, follow-up, patient identifiers."
session_key = AESGCM.generate_key(bit_length=256)  # Fresh symmetric key for this file.
nonce = os.urandom(12)  # Fresh AES-GCM nonce.
aesgcm = AESGCM(session_key)
ciphertext = aesgcm.encrypt(nonce, clinical_file, b"recipient=hospital")  # Encrypt and authenticate.

wrapped_key = hospital_public.encrypt(
    session_key,
    # OAEP is modern RSA encryption padding for wrapping small secrets such as session keys.
    padding.OAEP(mgf=padding.MGF1(algorithm=hashes.SHA256()), algorithm=hashes.SHA256(), label=None),
)

unwrapped_key = hospital_private.decrypt(
    wrapped_key,
    # Only the hospital private key can recover the session key.
    padding.OAEP(mgf=padding.MGF1(algorithm=hashes.SHA256()), algorithm=hashes.SHA256(), label=None),
)
recovered = AESGCM(unwrapped_key).decrypt(nonce, ciphertext, b"recipient=hospital")  # Verify and decrypt.
print(recovered)